# 04 — Real-Time Inference

This notebook runs the trained model against a **live webcam feed**, overlaying the predicted
gesture label on each frame in real time.

**Pipeline step:** Webcam → MediaPipe → Landmark Features → Trained Model → On-screen Label

> **Preferred method:** Run this as a standalone script for better performance:
> ```bash
> python src/inference.py
> # or, for an external webcam:
> python src/inference.py --camera 1
> ```

Press **`Q`** in the OpenCV window to quit.

## 4.1 Imports & Load Model

In [ ]:
import os
import pickle
import sys

import cv2
import mediapipe as mp
import numpy as np

sys.path.insert(0, os.path.abspath('..'))
from src.config import CAMERA_INDEX, LABELS_DICT, MODEL_PATH, MP_DETECTION_CONFIDENCE

# Load trained model
with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)['model']

print(f'Model loaded from: {MODEL_PATH}')
print(f'Label mapping    : {LABELS_DICT}')

## 4.2 Set Up MediaPipe

We use `static_image_mode=False` here (streaming mode) which is optimised for
consecutive video frames — it tracks the hand once detected rather than
re-detecting on every frame, giving lower latency.

In [ ]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(
    static_image_mode=False,          # streaming mode for lower latency
    min_detection_confidence=MP_DETECTION_CONFIDENCE
)

print('MediaPipe Hands initialised in streaming mode.')

## 4.3 Run Real-Time Inference Loop

The loop:
1. Reads a frame from the webcam.
2. Runs MediaPipe to detect hand landmarks.
3. Normalises landmarks and feeds them to the classifier.
4. Draws the predicted label and bounding box on the frame.
5. Displays the annotated frame — press **`Q`** to exit.

In [ ]:
cap = cv2.VideoCapture(CAMERA_INDEX)

print('Starting inference — press Q to quit.')

while True:
    ret, frame = cap.read()
    if not ret:
        print('[WARN] Failed to grab frame.')
        break

    h, w, _ = frame.shape
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(frame_rgb)

    if results.multi_hand_landmarks:
        data_aux = []
        x_coords, y_coords = [], []

        # Draw skeleton
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                frame, hand_landmarks, mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style()
            )

        # Extract normalised features
        for hand_landmarks in results.multi_hand_landmarks:
            for lm in hand_landmarks.landmark:
                x_coords.append(lm.x)
                y_coords.append(lm.y)
            min_x, min_y = min(x_coords), min(y_coords)
            for lm in hand_landmarks.landmark:
                data_aux.append(lm.x - min_x)
                data_aux.append(lm.y - min_y)

        # Bounding box
        x1 = max(int(min(x_coords) * w) - 20, 0)
        y1 = max(int(min(y_coords) * h) - 20, 0)
        x2 = min(int(max(x_coords) * w) + 20, w)
        y2 = min(int(max(y_coords) * h) + 20, h)

        try:
            prediction = model.predict([np.asarray(data_aux)])
            label = LABELS_DICT.get(int(prediction[0]), str(prediction[0]))
        except Exception:
            label = '?'

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 200, 100), 2)
        cv2.putText(frame, label, (x1, y1 - 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 200, 100), 3, cv2.LINE_AA)

    # HUD
    cv2.putText(frame, 'Hand Sign Recognition  |  Q to quit',
                (10, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1, cv2.LINE_AA)

    cv2.imshow('Hand Sign Recognition', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
hands.close()
print('Inference stopped.')